# Comex-stats — Colab Setup

**Run this notebook first** before opening any analysis notebook (`01_data_profile`, `06_china_imports`, etc.).

Expected files in your Colab session:
- `/content/Imports DATA/IMP_2025.csv`
- `/content/Imports DATA/IMP_2026.csv`
- `/content/Reference/NCM.csv`, `PAIS.csv`, `URF.csv`, `VIA.csv`, `UF.csv`

This notebook runs all four pipeline stages inline and writes parquet files to `/content/outputs/data/`.

## Setup — directories and imports

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Colab paths — match your actual uploaded file locations
IMPORTS_DIR = Path("/content/Imports DATA")
REFS_DIR    = Path("/content/Reference")
OUT_DIR     = Path("/content/outputs/data")
CHART_DIR   = Path("/content/outputs/charts")

OUT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

print("Output directories ready:")
print(f"  {OUT_DIR}")
print(f"  {CHART_DIR}")

# Verify source files are present
required_imports = [
    IMPORTS_DIR / "IMP_2025.csv",
    IMPORTS_DIR / "IMP_2026.csv",
]
required_refs = ["NCM.csv", "PAIS.csv", "URF.csv", "VIA.csv", "UF.csv"]

missing = []
for p in required_imports:
    if not p.exists():
        missing.append(str(p))
for f in required_refs:
    if not (REFS_DIR / f).exists():
        missing.append(str(REFS_DIR / f))

if missing:
    print("\n⚠️  Missing files — upload these to Colab before continuing:")
    for m in missing:
        print(f"  {m}")
else:
    print("\n✓ All source files found.")

## Stage 0 — Ingest

In [ ]:
STRING_COLS       = ["CO_NCM", "CO_UNID", "CO_PAIS", "SG_UF_NCM", "CO_VIA", "CO_URF"]
NUMERIC_INT_COLS  = ["CO_ANO", "CO_MES"]
NUMERIC_FLOAT_COLS = ["QT_ESTAT", "KG_LIQUIDO", "VL_FOB", "VL_FRETE", "VL_SEGURO"]
BUSINESS_KEY      = ["CO_ANO", "CO_MES", "CO_NCM", "CO_PAIS", "SG_UF_NCM", "CO_VIA", "CO_URF"]


def _detect_encoding(path: Path) -> str:
    for enc in ("utf-8-sig", "latin-1", "cp1252"):
        try:
            with open(path, encoding=enc) as f:
                f.readline()
            return enc
        except UnicodeDecodeError:
            continue
    return "latin-1"


def read_csv(path: Path, year: int) -> pd.DataFrame:
    enc = _detect_encoding(path)
    print(f"  Reading {path.name} (encoding={enc}) ...")
    # engine="python" + on_bad_lines="skip" handles quoting properly
    # AND skips IMP_2026.csv's malformed row (unmatched quote at row 572704).
    df = pd.read_csv(
        path,
        sep=";",
        encoding=enc,
        dtype={c: str for c in STRING_COLS},
        quotechar='"',
        on_bad_lines="skip",
        engine="python",
    )
    # Normalize column names (strip whitespace + any literal quote chars)
    df.columns = df.columns.str.strip().str.strip('"')
    # Strip whitespace and stray quotes from all object columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip().str.strip('"')
    # Cast numeric columns
    for col in NUMERIC_INT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in NUMERIC_FLOAT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].str.replace(",", ".", regex=False) if df[col].dtype == object else df[col], errors="coerce")
    df["SOURCE_YEAR"] = year
    print(f"    {len(df):,} rows loaded")
    return df


print("=" * 60)
print("  STAGE 0 — INGEST")
print("=" * 60)

frames = []
for year in (2025, 2026):
    p = IMPORTS_DIR / f"IMP_{year}.csv"
    if p.exists():
        frames.append(read_csv(p, year))
    else:
        print(f"  Skipping {p.name} — file not found")

if not frames:
    raise FileNotFoundError(f"No CSV files found in {IMPORTS_DIR}")

raw = pd.concat(frames, ignore_index=True)
print(f"\nCombined: {len(raw):,} rows, {len(raw.columns)} columns")
print(f"  Columns: {list(raw.columns)}")

# Basic profile
dup_keys = raw.duplicated(subset=BUSINESS_KEY).sum()
bad_ncm  = (~raw["CO_NCM"].str.replace(r"\D", "", regex=True).str.len().eq(8)).sum()
print(f"  Duplicate business keys : {dup_keys:,}")
print(f"  Bad NCM format rows     : {bad_ncm:,}")
for col in ["VL_FOB", "KG_LIQUIDO"]:
    if col in raw.columns:
        null_pct = 100 * raw[col].isna().sum() / len(raw)
        print(f"  {col} null rate         : {null_pct:.2f}%")

out_path = OUT_DIR / "raw_combined.parquet"
pq.write_table(pa.Table.from_pandas(raw, preserve_index=False), out_path, compression="snappy")
print(f"\n✓ raw_combined.parquet written ({out_path.stat().st_size/1e6:.1f} MB)")

## Stage 1 — Clean

In [ ]:
# 2026 is a partial year — only 2025 months are marked complete
KNOWN_COMPLETE_PERIODS = {(2025, m) for m in range(1, 13)}


def validate_ncm(df):
    digits_only = df["CO_NCM"].str.replace(r"\D", "", regex=True)
    valid_mask = digits_only.str.len() == 8
    return df[valid_mask].copy(), df[~valid_mask].copy()


def flag_zero_weight(df):
    df["is_zero_weight"] = (df["KG_LIQUIDO"] == 0) | (df["KG_LIQUIDO"].isna())
    return df


def flag_outliers(df):
    df["UNIT_FOB_PER_KG"] = np.where(
        df["KG_LIQUIDO"] > 0,
        df["VL_FOB"] / df["KG_LIQUIDO"],
        np.nan,
    )
    df["CO_POSICAO"] = df["CO_NCM"].str[:4]
    df["is_outlier_unitval"] = False
    has_price = df["UNIT_FOB_PER_KG"].notna() & (df["VL_FOB"] > 0)

    def iqr_bounds(series):
        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        iqr = q3 - q1
        return (series < q1 - 3 * iqr) | (series > q3 + 3 * iqr)

    outlier_mask = (
        df[has_price]
        .groupby("CO_POSICAO")["UNIT_FOB_PER_KG"]
        .transform(iqr_bounds)
        .reindex(df.index, fill_value=False)
        .astype(bool)
    )
    df["is_outlier_unitval"] = outlier_mask
    return df


def flag_period_completeness(df):
    df["is_complete_period"] = df.apply(
        lambda r: (int(r["CO_ANO"]), int(r["CO_MES"])) in KNOWN_COMPLETE_PERIODS
        if pd.notna(r["CO_ANO"]) and pd.notna(r["CO_MES"])
        else False,
        axis=1,
    )
    return df


print("=" * 60)
print("  STAGE 1 — CLEAN")
print("=" * 60)

raw = pd.read_parquet(OUT_DIR / "raw_combined.parquet")
print(f"  Loaded {len(raw):,} rows")

valid, invalid = validate_ncm(raw)
print(f"  Valid NCM rows   : {len(valid):,}")
print(f"  Invalid NCM rows : {len(invalid):,}")

valid = flag_zero_weight(valid)
valid = flag_outliers(valid)
print("  Flagging period completeness (may take a moment) ...")
valid = flag_period_completeness(valid)

out_count  = valid["is_outlier_unitval"].sum()
zero_w     = valid["is_zero_weight"].sum()
incomplete = (~valid["is_complete_period"]).sum()
print(f"  Outlier unit-value flags : {out_count:,} ({100*out_count/len(valid):.2f}%)")
print(f"  Zero-weight flags        : {zero_w:,} ({100*zero_w/len(valid):.2f}%)")
print(f"  Incomplete-period rows   : {incomplete:,} ({100*incomplete/len(valid):.2f}%)")

pq.write_table(pa.Table.from_pandas(valid, preserve_index=False), OUT_DIR / "clean.parquet", compression="snappy")
if len(invalid) > 0:
    pq.write_table(pa.Table.from_pandas(invalid, preserve_index=False), OUT_DIR / "invalid_ncm.parquet", compression="snappy")
    print(f"  invalid_ncm.parquet written ({len(invalid):,} rows)")

print(f"\n✓ clean.parquet written ({(OUT_DIR / 'clean.parquet').stat().st_size/1e6:.1f} MB)")

## Stage 2 — Enrich

In [ ]:
def _read_ref(filename, string_cols, **kwargs):
    path = REFS_DIR / filename
    for enc in ("utf-8-sig", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(
                path, sep=";", encoding=enc,
                dtype={c: str for c in string_cols},
                quotechar='"', low_memory=False, **kwargs,
            )
            for col in df.select_dtypes(include="object").columns:
                df[col] = df[col].str.strip().str.strip('"')
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Cannot decode {path}")


def load_ncm():
    df = _read_ref("NCM.csv", string_cols=["CO_NCM", "CO_SH6", "CO_PPE", "CO_FAT_AGREG"])
    return df[["CO_NCM", "CO_SH6", "NO_NCM_POR", "NO_NCM_ING", "CO_PPE", "CO_FAT_AGREG"]].drop_duplicates("CO_NCM")


def load_pais():
    df = _read_ref("PAIS.csv", string_cols=["CO_PAIS", "CO_PAIS_ISON3", "CO_PAIS_ISOA3"])
    return df[["CO_PAIS", "CO_PAIS_ISON3", "CO_PAIS_ISOA3", "NO_PAIS", "NO_PAIS_ING"]].drop_duplicates("CO_PAIS")


def load_urf():
    df = _read_ref("URF.csv", string_cols=["CO_URF"])
    return df[["CO_URF", "NO_URF"]].drop_duplicates("CO_URF")


def load_via():
    df = _read_ref("VIA.csv", string_cols=["CO_VIA"])
    return df[["CO_VIA", "NO_VIA"]].drop_duplicates("CO_VIA")


def load_uf():
    df = _read_ref("UF.csv", string_cols=["CO_UF", "SG_UF"])
    cols = [c for c in df.columns if c in ("SG_UF", "CO_UF", "NO_UF", "NO_REGIAO")]
    return df[cols].drop_duplicates(subset=[c for c in ("SG_UF", "CO_UF") if c in df.columns])


def join_and_report(df, ref, left_on, ref_name):
    merged = df.merge(ref, left_on=left_on, right_on=ref.columns[0], how="left")
    unmatched = merged[ref.columns[1]].isna().sum()
    pct = 100 * unmatched / max(len(df), 1)
    print(f"  {ref_name:20s}: {unmatched:,} unmatched ({pct:.1f}%)")
    return merged


def add_derived_fields(df):
    df["VL_CIF"] = df["VL_FOB"] + df["VL_FRETE"].fillna(0) + df["VL_SEGURO"].fillna(0)
    mask_kg = df["KG_LIQUIDO"] > 0
    df["UNIT_FOB_PER_KG"] = np.where(mask_kg, df["VL_FOB"] / df["KG_LIQUIDO"], np.nan)
    df["UNIT_CIF_PER_KG"] = np.where(mask_kg, df["VL_CIF"] / df["KG_LIQUIDO"], np.nan)
    df["FREIGHT_PER_KG"]  = np.where(mask_kg & df["VL_FRETE"].notna(), df["VL_FRETE"] / df["KG_LIQUIDO"], np.nan)
    df["FREIGHT_PCT_FOB"] = np.where(df["VL_FOB"] > 0, df["VL_FRETE"].fillna(0) / df["VL_FOB"], np.nan)
    df["CO_CAPITULO"] = df["CO_NCM"].str[:2]
    df["CO_POSICAO"]  = df["CO_NCM"].str[:4]
    df["PERIODO"] = (df["CO_ANO"].astype("Int64") * 100 + df["CO_MES"].astype("Int64")).astype("Int64")
    return df


print("=" * 60)
print("  STAGE 2 — ENRICH")
print("=" * 60)

df = pd.read_parquet(OUT_DIR / "clean.parquet")
print(f"  Loaded {len(df):,} rows")

print("\nLoading reference tables ...")
ncm_ref  = load_ncm()
pais_ref = load_pais()
urf_ref  = load_urf()
via_ref  = load_via()
uf_ref   = load_uf()

print("\nJoin match rates:")
df = join_and_report(df, ncm_ref,  left_on="CO_NCM",    ref_name="NCM")
df = join_and_report(df, pais_ref, left_on="CO_PAIS",   ref_name="PAIS")
df = join_and_report(df, urf_ref,  left_on="CO_URF",    ref_name="URF")
df = join_and_report(df, via_ref,  left_on="CO_VIA",    ref_name="VIA")
df = join_and_report(df, uf_ref,   left_on="SG_UF_NCM", ref_name="UF")

print("\nAdding derived fields ...")
df = add_derived_fields(df)

pq.write_table(pa.Table.from_pandas(df, preserve_index=False), OUT_DIR / "enriched.parquet", compression="snappy")
print(f"\n✓ enriched.parquet written ({(OUT_DIR / 'enriched.parquet').stat().st_size/1e6:.1f} MB)")
print(f"  {len(df):,} rows, {len(df.columns)} columns")

## Stage 3 — Analytical Marts

In [ ]:
AGGS_BASE = {"KG_LIQUIDO": "sum", "VL_FOB": "sum", "VL_CIF": "sum", "VL_FRETE": "sum"}


def p10(s): return float(s.quantile(0.10)) if s.notna().any() else np.nan
def p90(s): return float(s.quantile(0.90)) if s.notna().any() else np.nan


def unit_price_aggs(group_df, price_col="UNIT_FOB_PER_KG"):
    clean = group_df.loc[~group_df["is_outlier_unitval"], price_col].dropna()
    return {
        f"MEDIAN_{price_col}": float(clean.median()) if len(clean) else np.nan,
        f"P10_{price_col}":    p10(clean),
        f"P90_{price_col}":    p90(clean),
    }


def freight_aggs(group_df):
    frt_kg  = group_df["FREIGHT_PER_KG"].dropna()
    frt_pct = group_df["FREIGHT_PCT_FOB"].dropna()
    return {
        "MEDIAN_FREIGHT_PER_KG":  float(frt_kg.median())  if len(frt_kg)  else np.nan,
        "MEDIAN_FREIGHT_PCT_FOB": float(frt_pct.median()) if len(frt_pct) else np.nan,
    }


def build_mart(df, group_cols, name, include_freight=False):
    groups  = df.groupby(group_cols, observed=True)
    agg_df  = groups.agg(AGGS_BASE).reset_index()
    agg_df.columns = group_cols + list(AGGS_BASE.keys())
    agg_df["N_OPS"] = groups.size().values

    price_rows = []
    for key, grp in groups:
        row = {c: v for c, v in zip(group_cols, (key if isinstance(key, tuple) else (key,)))}
        row.update(unit_price_aggs(grp))
        if include_freight:
            row.update(freight_aggs(grp))
        price_rows.append(row)

    price_df = pd.DataFrame(price_rows)
    result = agg_df.merge(price_df, on=group_cols, how="left")
    print(f"  {name:45s}: {len(result):,} rows")
    return result


def write_mart(df, filename):
    path = OUT_DIR / filename
    pq.write_table(pa.Table.from_pandas(df, preserve_index=False), path, compression="snappy")


print("=" * 60)
print("  STAGE 3 — ANALYTICAL MARTS")
print("=" * 60)

df = pd.read_parquet(OUT_DIR / "enriched.parquet")
print(f"  Loaded {len(df):,} rows\n")

m = build_mart(df, ["CO_CAPITULO", "CO_ANO", "CO_MES"], "mart_chapter_month")
write_mart(m, "mart_chapter_month.parquet")

m = build_mart(df, ["CO_POSICAO", "CO_PAIS", "NO_PAIS_ING"], "mart_ncm4_country")
write_mart(m, "mart_ncm4_country.parquet")

m = build_mart(df, ["CO_POSICAO", "CO_URF", "NO_URF"], "mart_ncm4_urf")
write_mart(m, "mart_ncm4_urf.parquet")

m = build_mart(df, ["CO_POSICAO", "SG_UF_NCM"], "mart_ncm4_uf")
write_mart(m, "mart_ncm4_uf.parquet")

m = build_mart(df, ["CO_SH6", "CO_VIA", "NO_VIA"], "mart_sh6_via (freight)", include_freight=True)
write_mart(m, "mart_sh6_via.parquet")

m = build_mart(df, ["CO_POSICAO", "CO_ANO", "CO_MES"], "mart_ncm4_month")
write_mart(m, "mart_ncm4_month.parquet")

m = build_mart(df, ["CO_CAPITULO", "CO_PAIS", "NO_PAIS_ING", "CO_URF", "NO_URF", "SG_UF_NCM"], "mart_tradeline")
write_mart(m, "mart_tradeline.parquet")

print("\n✓ All 7 mart files written.")

## Verification

In [ ]:
expected_files = [
    "raw_combined.parquet",
    "clean.parquet",
    "enriched.parquet",
    "mart_chapter_month.parquet",
    "mart_ncm4_country.parquet",
    "mart_ncm4_urf.parquet",
    "mart_ncm4_uf.parquet",
    "mart_sh6_via.parquet",
    "mart_ncm4_month.parquet",
    "mart_tradeline.parquet",
]

print("=" * 60)
print("  PIPELINE VERIFICATION")
print("=" * 60)
print()

all_ok = True
for f in expected_files:
    p = OUT_DIR / f
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f"  ✓  {f:40s} {size_mb:6.1f} MB")
    else:
        print(f"  ✗  {f:40s} MISSING")
        all_ok = False

print()

# Spot-checks on enriched
enriched = pd.read_parquet(OUT_DIR / "enriched.parquet")
china_rows = enriched[enriched["CO_PAIS"] == "160"]
ch30 = (enriched
        .groupby("CO_CAPITULO")["VL_FOB"].sum()
        .sort_values(ascending=False)
        .head(10)
        .index.tolist())

checks = {
    "Row count > 1M"              : len(enriched) > 1_000_000,
    "China (CO_PAIS=160) present" : len(china_rows) > 0,
    "CO_SH6 coverage > 95%"       : (100 * enriched["CO_SH6"].notna().sum() / len(enriched)) > 95,
    "Chapter 30 (pharma) in top-10": "30" in ch30,
    "No negative VL_FOB"          : int((enriched["VL_FOB"] < 0).sum()) == 0,
}

for label, passed in checks.items():
    status = "✓" if passed else "✗"
    print(f"  [{status}] {label}")
    if not passed:
        all_ok = False

print()
if all_ok:
    print("  ✅  ALL CHECKS PASSED — open 01_data_profile.ipynb or 06_china_imports.ipynb to begin analysis.")
else:
    print("  ❌  SOME CHECKS FAILED — review the output above before proceeding.")

print(f"\n  China rows  : {len(china_rows):,} ({100*len(china_rows)/len(enriched):.1f}% of total)")
print(f"  Total rows  : {len(enriched):,}")